In [10]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.optim.adam import Adam
from torch.utils.data.dataloader import DataLoader
from torchvision.datasets.cifar import CIFAR10
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

# Skip-connections: ResNet

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super().__init__()

        self.c1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=kernel_size, stride=1, padding=1
        )
        self.bn1 = nn.BatchNorm2d(num_features=out_channels)
        self.relu = nn.ReLU()

        self.c2 = nn.Conv2d(
            out_channels, out_channels, kernel_size=kernel_size, stride=1, padding=1
        )
        self.bn2 = nn.BatchNorm2d(num_features=out_channels)

        self.downsample = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        identity = x  # skip-connection을 위함

        x = self.c1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.c2(x)
        x = self.bn2(x)

        identity = self.downsample(
            identity
        )  # skip connection할 때의 채널 수를 맞춰주기 위함

        x += identity  # skip-connection   ***핵심***
        x = self.relu(x)

        return x


class ResNet(nn.Module):
    def __init__(self, num_classes=10) -> None:
        super().__init__()

        self.b1 = BasicBlock(in_channels=3, out_channels=64)
        self.b2 = BasicBlock(in_channels=64, out_channels=128)
        self.b3 = BasicBlock(in_channels=128, out_channels=256)

        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(4096, 2048)
        self.fc2 = nn.Linear(2048, 512)
        self.fc3 = nn.Linear(512, num_classes)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.b1(x)
        x = self.pool(x)
        x = self.b2(x)
        x = self.pool(x)
        x = self.b3(x)
        x = self.pool(x)

        x = torch.flatten(x, start_dim=1)

        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)

        return x

In [ ]:
transforms = {
    'train': T.Compose(
        [
            T.RandomCrop((32, 32), padding=4),
            T.RandomHorizontalFlip(p=0.5),
            T.ToTensor(),
            T.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.247, 0.243, 0.261)),
        ]
    ),
    'val': T.Compose(
        [
            T.ToTensor(),
            T.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.247, 0.243, 0.261)),
        ]
    ),
}

In [13]:
training_data = CIFAR10(
    root='../data/images/', train=True, download=True, transform=transforms['train']
)
test_data = CIFAR10(
    root='../data/images/', train=False, download=True, transform=transforms['val']
)

train_loader = DataLoader(training_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

c:\Users\hanvv\OneDrive\문서\Study\potenup3\dl-with-pytorch\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [14]:
model = ResNet(num_classes=10)
model.to(device)

ResNet(
  (b1): BasicBlock(
    (c1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (c2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (downsample): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
  )
  (b2): BasicBlock(
    (c1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (c2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (downsample): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1))
  )
  (b3): BasicBlock(
    (c1): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))


In [15]:
lr = 1e-4
num_epochs = 30

optim = Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()
model.train()

for epoch in range(num_epochs):
    running_loss = 0.0

    iterator = tqdm(train_loader, leave=True)

    for data, label in iterator:
        data, label = data.to(device), label.to(device)

        optim.zero_grad()
        preds = model(data)
        loss = criterion(preds, label)
        loss.backward()
        optim.step()

        running_loss += loss.item()

        iterator.set_description(f'epoch:{epoch + 1} loss:{loss.item():.4f}')
    print(f'epoch {epoch + 1}  mean loss: {running_loss / len(train_loader):.4f}')

torch.save(model.state_dict(), '../data/models/resnet-CIFAR.pth')

epoch:1 loss:0.8916: 100%|██████████| 1563/1563 [00:15<00:00, 100.32it/s]


epoch 1  mean loss: 1.3380


epoch:2 loss:0.5187: 100%|██████████| 1563/1563 [00:15<00:00, 99.10it/s] 


epoch 2  mean loss: 0.8877


epoch:3 loss:0.4068: 100%|██████████| 1563/1563 [00:15<00:00, 99.31it/s] 


...

epoch 28  mean loss: 0.0921


epoch:29 loss:0.0570: 100%|██████████| 1563/1563 [00:14<00:00, 104.65it/s]


epoch 29  mean loss: 0.0876


epoch:30 loss:0.0048: 100%|██████████| 1563/1563 [00:15<00:00, 98.98it/s] 


epoch 30  mean loss: 0.0837


In [16]:
model.load_state_dict(
    torch.load('../data/models/resnet-CIFAR.pth', map_location=device)
)

model.eval()

num_corr = 0

with torch.no_grad():
    for data, label in test_loader:
        data, label = data.to(device), label.to(device)

        preds = model(data)
        preds = torch.argmax(preds, dim=1)

        num_corr += (preds == label).sum().item()

print(f'accuracy: {num_corr / len(test_data):.4f}')

accuracy: 0.8932
